# Week 6 – Feature Engineering and Market Metrics

**Purpose:** Create the engineered market metrics required for the IDX Exchange Week 6 deliverable and prepare output files for future Tableau dashboard development.

This notebook is organized into two sections:

1. **Sold data metrics** – used for closed-sales analysis, pricing ratios, days on market, and transaction timeline metrics.
2. **Listing data metrics** – used for new-listing activity, listing-side price per square foot, and market supply summaries.

**Main deliverables created in this notebook:**

- Engineered sold dataset
- Engineered listing dataset
- Sample output tables showing new columns populated correctly
- Segmented summary tables by county, property type/subtype, MLS area, and office

## 1. Setup: Import Packages and Define File Paths

This section imports the required Python packages, defines the project folder, identifies the input files from the previous weeks, and creates the Week 6 output folder.

> Update `SOLD_FILE` and `LISTINGS_FILE` only if your actual file names are different.

In [29]:
from pathlib import Path
import pandas as pd
import numpy as np

# Base project directory
BASE_DIR = Path("/Users/amyliu/Desktop/IDX")

# Input files from previous cleaned / enriched outputs
SOLD_FILE = BASE_DIR / "data" / "generated" / "sold_with_rates_week4-5.csv"
LISTINGS_FILE = BASE_DIR / "data" / "generated" / "listing_with_rates_week4-5.csv"

# Week 6 output folder
OUTPUT_DIR = BASE_DIR / "data" / "generated" / "week6"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Sold input file:", SOLD_FILE)
print("Listings input file:", LISTINGS_FILE)
print("Output directory:", OUTPUT_DIR)

if not SOLD_FILE.exists():
    raise FileNotFoundError(f"Sold file not found: {SOLD_FILE}")

if not LISTINGS_FILE.exists():
    raise FileNotFoundError(f"Listings file not found: {LISTINGS_FILE}")

Sold input file: /Users/amyliu/Desktop/IDX/data/generated/sold_with_rates_week4-5.csv
Listings input file: /Users/amyliu/Desktop/IDX/data/generated/listing_with_rates_week4-5.csv
Output directory: /Users/amyliu/Desktop/IDX/data/generated/week6


## 2. Load Input Datasets

The sold dataset is used to create closed-sales metrics. The listing dataset is used to create new-listing and market activity metrics.

In [35]:
sold = pd.read_csv(SOLD_FILE, low_memory=False)

print("Sold dataset")
print(f"Rows loaded: {len(sold):,}")
print(f"Columns loaded: {sold.shape[1]:,}")

sold.head()

Sold dataset
Rows loaded: 591,733
Columns loaded: 84


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,ListAgentFirstName,...,invalid_dom_flag,invalid_bedrooms_flag,invalid_bathrooms_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coord_flag,zero_coord_flag,positive_longitude_flag,implausible_coord_flag
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,False,499000.0,551985747,2024-01-26,240000.0,Joan,...,False,False,False,False,False,False,True,False,False,False
1,HighDesert,HighDesert,NaN,NaN,NaN,0.0,535486633,2024-01-24,950.0,Elizabeth,...,False,False,False,False,False,False,False,False,False,False
2,OrangeCounty,OrangeCounty,NaN,True,NaN,75000.0,529986282,2024-01-16,45000.0,Joseph,...,False,False,False,False,False,False,False,False,False,False
3,InlandValleys,InlandValleys,NaN,True,NaN,199000.0,529618166,2024-01-08,141500.0,CAROL,...,False,False,False,False,False,False,False,False,False,False
4,SouthwestRiversideCounty,SouthwestRiversideCounty,NaN,True,NaN,19500.0,522614340,2024-01-17,15000.0,Jeremie,...,False,False,False,False,False,False,False,False,False,False


In [36]:
print(sold["PropertyType"].value_counts(dropna=False))

PropertyType
Residential            397603
ResidentialLease       135617
Land                    19345
ManufacturedInPark      16082
ResidentialIncome       15865
CommercialSale           3714
CommercialLease          3104
BusinessOpportunity       403
Name: count, dtype: int64


In [37]:
# Filter to Residential only
before_residential_filter = len(sold)

sold = sold[sold["PropertyType"] == "Residential"].copy()

after_residential_filter = len(sold)

print(f"\nRows before Residential filter: {before_residential_filter:,}")
print(f"Rows after Residential filter: {after_residential_filter:,}")
print(f"Rows removed by Residential filter: {before_residential_filter - after_residential_filter:,}")

print("\nPropertyType distribution after Residential filter:")
print(sold["PropertyType"].value_counts(dropna=False))


Rows before Residential filter: 591,733
Rows after Residential filter: 397,603
Rows removed by Residential filter: 194,130

PropertyType distribution after Residential filter:
PropertyType
Residential    397603
Name: count, dtype: int64


In [31]:
listings = pd.read_csv(LISTINGS_FILE, low_memory=False)

print("Listing dataset")
print(f"Rows loaded: {len(listings):,}")
print(f"Columns loaded: {listings.shape[1]:,}")

listings.head()

Listing dataset
Rows loaded: 852,963
Columns loaded: 74


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,invalid_dom_flag,invalid_bedrooms_flag,invalid_bathrooms_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coord_flag,zero_coord_flag,positive_longitude_flag,implausible_coord_flag
0,90000.0,1075010398,miriamlara03@gmail.com,NaN,NaN,Miriam,Lara,34.097939,-117.909653,1045 N Azusa 61,...,False,False,False,False,False,False,False,False,False,False
1,1500000.0,1074974457,janelle@judsonre.com,NaN,NaN,Janelle,Judson,33.121241,-117.081614,NaN,...,False,False,False,False,False,False,False,False,False,False
2,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,False,False,False,False,False,False,False,False,False,False
3,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,False,False,False,False,False,False,False,False,False,False
4,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,False,False,False,False,False,False,False,False,False,False


In [38]:
# Confirm property type distribution before filtering
print("PropertyType distribution before Residential filter:")
print(listings["PropertyType"].value_counts(dropna=False))

# Filter to Residential only
before_residential_filter = len(listings)

listings = listings[listings["PropertyType"] == "Residential"].copy()

after_residential_filter = len(listings)

print(f"\nRows before Residential filter: {before_residential_filter:,}")
print(f"Rows after Residential filter: {after_residential_filter:,}")
print(f"Rows removed by Residential filter: {before_residential_filter - after_residential_filter:,}")

print("\nPropertyType distribution after Residential filter:")
print(listings["PropertyType"].value_counts(dropna=False))

PropertyType distribution before Residential filter:
PropertyType
Residential            540183
ResidentialLease       178041
Land                    56372
ResidentialIncome       31650
ManufacturedInPark      24531
CommercialSale          11715
CommercialLease          7736
BusinessOpportunity      2735
Name: count, dtype: int64

Rows before Residential filter: 852,963
Rows after Residential filter: 540,183
Rows removed by Residential filter: 312,780

PropertyType distribution after Residential filter:
PropertyType
Residential    540183
Name: count, dtype: int64


---

# Part A – Sold Data Feature Engineering

The sold dataset supports the official Week 6 metrics such as price ratio, price per square foot, close-to-original-list ratio, and transaction timeline metrics.

## 3. Check Required Sold Columns

Before creating features, this step verifies whether the required sold-side columns exist in the dataset.

In [39]:
sold_required_cols = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "DaysOnMarket",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "ListOfficeName",
    "BuyerOfficeName"
]

sold_column_check = pd.DataFrame({
    "column": sold_required_cols,
    "exists": [col in sold.columns for col in sold_required_cols]
})

sold_column_check

,column,exists
0,CloseDate,True
1,PurchaseContractDate,True
2,ListingContractDate,True
3,ClosePrice,True
4,OriginalListPrice,True
5,LivingArea,True
6,DaysOnMarket,True
7,PropertyType,True
8,PropertySubType,True
9,CountyOrParish,True


## 4. Convert Sold Date and Numeric Fields

The transaction timeline metrics require date fields to be in datetime format. Price and area fields need to be numeric before ratios can be calculated.

In [40]:
# Convert sold date columns
sold_date_cols = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

for col in sold_date_cols:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors="coerce")
        print(f"Converted to datetime: {col}")
    else:
        print(f"Missing date column: {col}")

# Convert sold numeric columns
sold_numeric_cols = [
    "ClosePrice",
    "OriginalListPrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in sold_numeric_cols:
    if col in sold.columns:
        sold[col] = pd.to_numeric(sold[col], errors="coerce")
        print(f"Converted to numeric: {col}")
    else:
        print(f"Missing numeric column: {col}")

Converted to datetime: CloseDate
Converted to datetime: PurchaseContractDate
Converted to datetime: ListingContractDate
Converted to datetime: ContractStatusChangeDate
Converted to numeric: ClosePrice
Converted to numeric: OriginalListPrice
Converted to numeric: ListPrice
Converted to numeric: LivingArea
Converted to numeric: DaysOnMarket


## 5. Define Safe Division Function

This helper function prevents invalid calculations when the denominator is missing or equal to zero.

In [41]:
def safe_divide(numerator, denominator):
    """
    Return numerator / denominator.
    If denominator is missing or zero, return NaN.
    """
    return np.where(
        denominator.notna() & (denominator != 0),
        numerator / denominator,
        np.nan
    )

## 6. Create Sold-Side Week 6 Metrics

This step creates the official Week 6 sold-side engineered metrics.

| Metric | Formula / Source | Purpose |
|---|---|---|
| `price_ratio` | `ClosePrice / OriginalListPrice` | Measures negotiation strength |
| `close_to_original_list_ratio` | `ClosePrice / OriginalListPrice` | Captures full price reduction history |
| `price_per_sqft` | `ClosePrice / LivingArea` | Normalizes price across different home sizes |
| `days_on_market` | `DaysOnMarket` | Measures time-to-sell |
| `close_year`, `close_month`, `yrmo` | Derived from `CloseDate` | Enables time-series analysis |
| `listing_to_contract_days` | `PurchaseContractDate - ListingContractDate` | Measures time from listing to accepted offer |
| `contract_to_close_days` | `CloseDate - PurchaseContractDate` | Measures escrow / closing period duration |

In [42]:
# Price Ratio = ClosePrice / OriginalListPrice
sold["price_ratio"] = safe_divide(
    sold["ClosePrice"],
    sold["OriginalListPrice"]
)

# Close-to-Original-List Ratio = ClosePrice / OriginalListPrice
sold["close_to_original_list_ratio"] = safe_divide(
    sold["ClosePrice"],
    sold["OriginalListPrice"]
)

# Price Per Sq Ft = ClosePrice / LivingArea
sold["price_per_sqft"] = safe_divide(
    sold["ClosePrice"],
    sold["LivingArea"]
)

# Days on Market
sold["days_on_market"] = sold["DaysOnMarket"]

# Year / Month / YrMo from CloseDate
sold["close_year"] = sold["CloseDate"].dt.year
sold["close_month"] = sold["CloseDate"].dt.month
sold["yrmo"] = sold["CloseDate"].dt.to_period("M").astype(str)

# Listing-to-Contract Days = PurchaseContractDate - ListingContractDate
sold["listing_to_contract_days"] = (
    sold["PurchaseContractDate"] - sold["ListingContractDate"]
).dt.days

# Contract-to-Close Days = CloseDate - PurchaseContractDate
sold["contract_to_close_days"] = (
    sold["CloseDate"] - sold["PurchaseContractDate"]
).dt.days

sold_engineered_cols = [
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "days_on_market",
    "close_year",
    "close_month",
    "yrmo",
    "listing_to_contract_days",
    "contract_to_close_days"
]

sold[sold_engineered_cols].head()

,price_ratio,close_to_original_list_ratio,price_per_sqft,days_on_market,close_year,close_month,yrmo,listing_to_contract_days,contract_to_close_days
0,0.480962,0.480962,210.526316,777,2024,1,2024-01,777.0,65.0
5,1.072510,1.072510,412.867275,33,2024,1,2024-01,114.0,919.0
9,1.094743,1.094743,410.334347,228,2024,1,2024-01,255.0,778.0
28,NaN,NaN,430.075188,0,2024,1,2024-01,188.0,-188.0
29,1.000000,1.000000,591.891046,0,2024,1,2024-01,0.0,0.0


## 7. Validate Sold Engineered Metrics

This section checks the missing-value pattern and distribution of the newly created sold-side metrics.

In [43]:
sold_metric_nulls = (
    sold[sold_engineered_cols]
    .isna()
    .sum()
    .reset_index()
)

sold_metric_nulls.columns = ["engineered_column", "null_count"]
sold_metric_nulls["null_pct"] = sold_metric_nulls["null_count"] / len(sold)

sold_metric_nulls

,engineered_column,null_count,null_pct
0,price_ratio,725,0.001823
1,close_to_original_list_ratio,725,0.001823
2,price_per_sqft,375,0.000943
3,days_on_market,0,0.000000
4,close_year,0,0.000000
5,close_month,0,0.000000
6,yrmo,0,0.000000
7,listing_to_contract_days,195,0.000490
8,contract_to_close_days,194,0.000488


In [44]:
sold_metric_summary_cols = [
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "days_on_market",
    "listing_to_contract_days",
    "contract_to_close_days"
]

sold_metric_summary = (
    sold[sold_metric_summary_cols]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99])
    .T
)

sold_metric_summary

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
price_ratio,396878.0,57.214015,15672.284550,0.0,0.953488,0.995607,1.019553,1.068063,1.116008,1.280505,9.118000e+06
close_to_original_list_ratio,396878.0,57.214015,15672.284550,0.0,0.953488,0.995607,1.019553,1.068063,1.116008,1.280505,9.118000e+06
price_per_sqft,397228.0,645.470982,5084.547292,0.0,368.378427,537.313433,732.248521,990.593814,1215.554046,1884.861243,1.164067e+06
days_on_market,397603.0,37.336788,53.539245,-288.0,8.000000,19.000000,48.000000,94.000000,131.000000,229.000000,1.243000e+04
listing_to_contract_days,397408.0,45.080046,87.315578,-36407.0,10.000000,25.000000,58.000000,109.000000,152.000000,272.000000,1.465700e+04
contract_to_close_days,397409.0,31.674965,63.460096,-331.0,21.000000,29.000000,36.000000,50.000000,64.000000,118.000000,3.662900e+04


## 8. Sold Sample Output Table

This sample table demonstrates that the required sold-side engineered columns were created and populated correctly.

In [45]:
sold_sample_cols = [
    "CloseDate",
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "DaysOnMarket",
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "close_year",
    "close_month",
    "yrmo",
    "ListingContractDate",
    "PurchaseContractDate",
    "listing_to_contract_days",
    "contract_to_close_days",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor"
]

sold_sample_cols_existing = [col for col in sold_sample_cols if col in sold.columns]
week6_sold_sample_output = sold[sold_sample_cols_existing].head(20)

week6_sold_sample_output

,CloseDate,ClosePrice,OriginalListPrice,LivingArea,DaysOnMarket,price_ratio,close_to_original_list_ratio,price_per_sqft,close_year,close_month,yrmo,ListingContractDate,PurchaseContractDate,listing_to_contract_days,contract_to_close_days,PropertyType,PropertySubType,CountyOrParish,MLSAreaMajor
0,2024-01-26,240000.0,499000.0,1140.0,777,0.480962,0.480962,210.526316,2024,1,2024-01,2021-10-06,2023-11-22,777.0,65.0,Residential,Condominium,San Mateo,699 - Not Defined
5,2024-01-05,815000.0,759900.0,1974.0,33,1.072510,1.072510,412.867275,2024,1,2024-01,2021-03-08,2021-06-30,114.0,919.0,Residential,SingleFamilyResidence,San Diego,91950 - National City
9,2024-01-05,810000.0,739900.0,1974.0,228,1.094743,1.094743,410.334347,2024,1,2024-01,2021-03-08,2021-11-18,255.0,778.0,Residential,SingleFamilyResidence,San Diego,91950 - National City
28,2024-01-30,858000.0,NaN,1995.0,0,NaN,NaN,430.075188,2024,1,2024-01,2024-01-30,2024-08-05,188.0,-188.0,Residential,SingleFamilyResidence,Riverside,NaN
29,2024-01-29,1890500.0,1890500.0,3194.0,0,1.000000,1.000000,591.891046,2024,1,2024-01,2024-01-29,2024-01-29,0.0,0.0,Residential,SingleFamilyResidence,Los Angeles,TAR - Tarzana
30,2024-01-02,2100000.0,2100000.0,3736.0,0,1.000000,1.000000,562.098501,2024,1,2024-01,2023-11-15,2023-11-15,0.0,48.0,Residential,SingleFamilyResidence,San Luis Obispo,SLO - San Luis Obispo
31,2024-01-22,1950000.0,1950000.0,2100.0,-36,1.000000,1.000000,928.571429,2024,1,2024-01,2023-11-17,2023-12-19,32.0,34.0,Residential,SingleFamilyResidence,Mendocino,FBS - Fort Bragg South
33,2024-01-31,2340000.0,NaN,2442.0,0,NaN,NaN,958.230958,2024,1,2024-01,2024-01-31,2024-03-04,33.0,-33.0,Residential,SingleFamilyResidence,Santa Clara,699 - Not Defined
35,2024-01-31,1485000.0,1550000.0,1601.0,0,0.958065,0.958065,927.545284,2024,1,2024-01,2024-01-09,2024-01-10,1.0,21.0,Residential,Condominium,San Diego,92101 - San Diego Downtown
37,2024-01-30,1130000.0,999000.0,2136.0,1,1.131131,1.131131,529.026217,2024,1,2024-01,2024-01-12,2024-01-13,1.0,17.0,Residential,Duplex,Los Angeles,C21 - Silver Lake - Echo Park


## 9. Sold Segmented Summary by County

This table satisfies the Week 6 requirement to include at least one segmented summary table grouped by `CountyOrParish` or `PropertyType`. It summarizes sales volume, prices, price per square foot, days on market, and transaction timeline metrics by county.

In [46]:
county_summary = (
    sold.groupby("CountyOrParish", dropna=False)
    .agg(
        closed_sales=("ClosePrice", "count"),
        median_close_price=("ClosePrice", "median"),
        average_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("price_per_sqft", "median"),
        average_days_on_market=("days_on_market", "mean"),
        median_days_on_market=("days_on_market", "median"),
        average_close_to_original_list_ratio=("close_to_original_list_ratio", "mean"),
        median_listing_to_contract_days=("listing_to_contract_days", "median"),
        median_contract_to_close_days=("contract_to_close_days", "median")
    )
    .reset_index()
    .sort_values("closed_sales", ascending=False)
)

county_summary.head(20)

,CountyOrParish,closed_sales,median_close_price,average_close_price,median_price_per_sqft,average_days_on_market,median_days_on_market,average_close_to_original_list_ratio,median_listing_to_contract_days,median_contract_to_close_days
20,Los Angeles,98836,900000.0,1.322352e+06,608.273492,36.660518,19.0,113.874677,25.0,30.0
37,Riverside,55027,600000.0,7.191414e+05,320.918367,48.256129,30.0,96.622733,38.0,29.0
41,San Diego,48974,895000.0,1.435684e+06,591.111111,29.529220,15.0,1.386839,21.0,26.0
31,Orange,44733,1175000.0,1.532290e+06,671.977447,31.429571,14.0,82.803089,25.0,29.0
40,San Bernardino,37129,530000.0,5.988718e+05,332.342742,45.931105,25.0,59.382398,32.0,32.0
0,Alameda,18752,1135000.0,1.309826e+06,701.051577,25.960004,14.0,1.203303,14.0,23.0
7,Contra Costa,18093,820000.0,1.129247e+06,520.464881,29.022550,15.0,1.061966,15.0,23.0
47,Santa Clara,17862,1600000.0,1.920275e+06,967.462160,21.900347,9.0,1.040986,9.0,25.0
60,Ventura,12324,866280.5,1.074604e+06,515.836136,43.208942,28.0,1.501203,35.0,24.0
45,San Mateo,7004,1700000.0,2.195950e+06,1050.088548,28.770988,12.0,2.815939,12.0,23.0


## 10. Sold Segmented Summary by Property Type and Subtype

This table supports later Tableau filtering by property category and helps compare pricing and market speed across property subtypes.

In [47]:
property_summary = (
    sold.groupby(["PropertyType", "PropertySubType"], dropna=False)
    .agg(
        closed_sales=("ClosePrice", "count"),
        median_close_price=("ClosePrice", "median"),
        average_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("price_per_sqft", "median"),
        average_days_on_market=("days_on_market", "mean"),
        average_close_to_original_list_ratio=("close_to_original_list_ratio", "mean")
    )
    .reset_index()
    .sort_values("closed_sales", ascending=False)
)

property_summary.head(20)

,PropertyType,PropertySubType,closed_sales,median_close_price,average_close_price,median_price_per_sqft,average_days_on_market,average_close_to_original_list_ratio
14,Residential,SingleFamilyResidence,297330,890000.0,1.287398e+06,532.189955,36.247168,63.876061
3,Residential,Condominium,65842,627500.0,8.784747e+05,565.522621,41.601592,26.746768
18,Residential,Townhouse,23345,805000.0,1.021873e+06,562.727273,31.933433,43.155612
9,Residential,ManufacturedOnLand,5124,323950.0,3.505442e+05,225.462650,58.063232,1.303678
5,Residential,Duplex,2161,912000.0,1.226184e+06,543.750000,42.865340,451.352974
15,Residential,StockCooperative,1601,360000.0,3.912889e+05,395.000000,37.726421,1.569826
20,Residential,NaN,776,810000.0,1.002689e+06,570.970696,44.802835,0.983890
1,Residential,Cabin,452,241250.0,2.833005e+05,291.542659,79.066372,0.897593
19,Residential,Triplex,325,1100000.0,1.299329e+06,463.381555,57.412308,1.031447
10,Residential,MixedUse,195,700000.0,1.011545e+06,433.660012,83.215385,0.888430


## 11. Sold Segmented Summary by MLS Area

This table provides a geographic market summary at the MLS area level.

In [48]:
mls_area_summary = (
    sold.groupby("MLSAreaMajor", dropna=False)
    .agg(
        closed_sales=("ClosePrice", "count"),
        median_close_price=("ClosePrice", "median"),
        average_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("price_per_sqft", "median"),
        average_days_on_market=("days_on_market", "mean"),
        average_close_to_original_list_ratio=("close_to_original_list_ratio", "mean")
    )
    .reset_index()
    .sort_values("closed_sales", ascending=False)
)

mls_area_summary.head(20)

,MLSAreaMajor,closed_sales,median_close_price,average_close_price,median_price_per_sqft,average_days_on_market,average_close_to_original_list_ratio
1088,NaN,53120,793000.0,1.042500e+06,524.394929,33.612025,1.593785
299,699 - Not Defined,41925,1225000.0,1.579595e+06,778.093498,30.918831,1.176117
957,SRCAR - Southwest Riverside County,19427,585000.0,6.403423e+05,293.283293,41.625521,76.011443
133,252 - Riverside,5038,656000.0,7.105548e+05,378.056503,38.389440,1.021901
128,248 - Corona,3218,757000.0,7.991408e+05,396.888661,38.739279,320.094074
676,LAC - Lancaster,2944,475000.0,4.902772e+05,270.975515,41.266644,1.664067
1039,VIC - Victorville,2813,439000.0,6.043444e+05,238.123167,41.993957,2.139389
146,274 - San Bernardino,2787,499134.0,5.069932e+05,352.331606,36.966272,2.309652
136,263 - Banning/Beaumont/Cherry Valley,2787,499900.0,4.933169e+05,259.840778,47.649085,343.791476
291,686 - Ontario,2527,650700.0,6.658477e+05,418.616480,33.976256,1.234250


## 12. Sold Competitive Summary by Listing Office and Buyer Office

This summary supports later competitive intelligence work by identifying office-level closed sales volume and transaction value.

In [49]:
office_summary = (
    sold.groupby(["ListOfficeName", "BuyerOfficeName"], dropna=False)
    .agg(
        closed_sales=("ClosePrice", "count"),
        total_sales_volume=("ClosePrice", "sum"),
        median_close_price=("ClosePrice", "median"),
        average_days_on_market=("days_on_market", "mean"),
        average_close_to_original_list_ratio=("close_to_original_list_ratio", "mean")
    )
    .reset_index()
    .sort_values("total_sales_volume", ascending=False)
)

office_summary.head(20)

,ListOfficeName,BuyerOfficeName,closed_sales,total_sales_volume,median_close_price,average_days_on_market,average_close_to_original_list_ratio
47997,Compass,Compass,7086,1.510951e+10,1585000.0,29.408270,1.148253
42813,Coldwell Banker Realty,Coldwell Banker Realty,3391,7.020896e+09,1305000.0,33.276320,1.284456
42836,Coldwell Banker Realty,Compass,1512,3.031104e+09,1522500.0,31.787698,1.004505
47967,Compass,Coldwell Banker Realty,1279,2.782996e+09,1591000.0,31.581704,1.690188
46202,Coldwell Banker West,Coldwell Banker West,432,2.131376e+09,825000.0,24.655093,5.511482
51211,Compass,NaN,817,1.465945e+09,1405000.0,27.427173,1.037595
100598,Keller Williams Realty,Keller Williams Realty,1343,1.460985e+09,920000.0,30.128071,1.007789
86312,Intero Real Estate Services,Intero Real Estate Services,836,1.436281e+09,1445444.0,22.083732,1.032043
70353,First Team Real Estate,First Team Real Estate,1086,1.349092e+09,1100000.0,24.271639,1.885348
15963,Berkshire Hathaway HomeServices California Pro...,Berkshire Hathaway HomeServices California Pro...,784,1.334192e+09,1092500.0,39.526786,0.969112


---

# Part B – Listing Data Feature Engineering

The listing dataset is used for market activity metrics, especially **new listings**, which will be needed in Tableau dashboards. Listing records do not always have sold-side fields such as `ClosePrice`, so the listing-side features focus on `ListingContractDate`, `ListPrice`, `LivingArea`, and listing activity summaries.

## 13. Check Required Listing Columns

This step verifies whether the required listing-side columns exist in the dataset.

In [50]:
listing_required_cols = [
    "ListingContractDate",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "ListOfficeName",
    "City",
    "PostalCode"
]

listing_column_check = pd.DataFrame({
    "column": listing_required_cols,
    "exists": [col in listings.columns for col in listing_required_cols]
})

listing_column_check

,column,exists
0,ListingContractDate,True
1,ListPrice,True
2,LivingArea,True
3,DaysOnMarket,True
4,PropertyType,True
5,PropertySubType,True
6,CountyOrParish,True
7,MLSAreaMajor,True
8,ListOfficeName,True
9,City,True


## 14. Convert Listing Date and Numeric Fields

The listing-side metrics require `ListingContractDate`, `ListPrice`, `LivingArea`, and `DaysOnMarket` to be in the correct data types.

In [51]:
# Convert listing date field
listings["ListingContractDate"] = pd.to_datetime(
    listings["ListingContractDate"],
    errors="coerce"
)

# Convert listing numeric fields
listing_numeric_cols = [
    "ListPrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in listing_numeric_cols:
    listings[col] = pd.to_numeric(listings[col], errors="coerce")
    print(f"Converted {col} to numeric")

Converted ListPrice to numeric
Converted LivingArea to numeric
Converted DaysOnMarket to numeric


## 15. Create Listing-Side Week 6 Metrics

Listing-side metrics focus on new-listing activity and listing price normalization.

| Metric | Formula / Source | Purpose |
|---|---|---|
| `list_year`, `list_month`, `list_yrmo` | Derived from `ListingContractDate` | Enables monthly new-listing analysis |
| `list_price_per_sqft` | `ListPrice / LivingArea` | Normalizes listing price across home sizes |
| `listing_days_on_market` | `DaysOnMarket` | Measures listing-side market exposure |

In [52]:
# Year / Month / YrMo from ListingContractDate
listings["list_year"] = listings["ListingContractDate"].dt.year
listings["list_month"] = listings["ListingContractDate"].dt.month
listings["list_yrmo"] = listings["ListingContractDate"].dt.to_period("M").astype(str)

# List Price Per Sq Ft = ListPrice / LivingArea
listings["list_price_per_sqft"] = safe_divide(
    listings["ListPrice"],
    listings["LivingArea"]
)

# Listing Days on Market
listings["listing_days_on_market"] = listings["DaysOnMarket"]

listing_engineered_cols = [
    "list_year",
    "list_month",
    "list_yrmo",
    "list_price_per_sqft",
    "listing_days_on_market"
]

listings[listing_engineered_cols].head()

,list_year,list_month,list_yrmo,list_price_per_sqft,listing_days_on_market
2,2024,1,2024-01,1029.976941,127
3,2024,1,2024-01,896.700143,1
4,2024,1,2024-01,969.230769,1
5,2024,1,2024-01,414.431330,0
6,2024,1,2024-01,5482.550625,2


## 16. Monthly New Listings Summary

This summary supports the Tableau dashboard requirement for tracking new listings over time.

In [53]:
monthly_new_listings = (
    listings.groupby("list_yrmo", dropna=False)
    .agg(
        new_listings=("ListingContractDate", "count"),
        median_list_price=("ListPrice", "median"),
        average_list_price=("ListPrice", "mean"),
        median_list_price_per_sqft=("list_price_per_sqft", "median"),
        average_listing_days_on_market=("listing_days_on_market", "mean")
    )
    .reset_index()
    .sort_values("list_yrmo")
)

monthly_new_listings.head(20)

,list_yrmo,new_listings,median_list_price,average_list_price,median_list_price_per_sqft,average_listing_days_on_market
0,2024-01,17007,799000.0,1.267894e+06,526.472997,29.501441
1,2024-02,17490,818900.0,1.266638e+06,539.932040,24.548885
2,2024-03,20501,848000.0,1.322351e+06,547.325103,19.835130
3,2024-04,24025,875000.0,1.362140e+06,564.763560,10.861644
4,2024-05,25447,886930.0,1.390613e+06,574.074074,11.028530
5,2024-06,23310,875000.0,1.343788e+06,564.347008,10.615959
6,2024-07,23019,850000.0,1.275324e+06,560.064123,10.264955
7,2024-08,22215,849000.0,1.267652e+06,553.724456,8.759757
8,2024-09,22257,850000.0,1.355915e+06,560.487255,11.985488
9,2024-10,21921,829000.0,1.274271e+06,543.976684,12.733589


## 17. Listing Segmented Summary by County

This table summarizes listing activity by county, including new-listing count, list price, list price per square foot, and listing days on market.

In [54]:
listing_county_summary = (
    listings.groupby("CountyOrParish", dropna=False)
    .agg(
        new_listings=("ListingContractDate", "count"),
        median_list_price=("ListPrice", "median"),
        average_list_price=("ListPrice", "mean"),
        median_list_price_per_sqft=("list_price_per_sqft", "median"),
        average_listing_days_on_market=("listing_days_on_market", "mean")
    )
    .reset_index()
    .sort_values("new_listings", ascending=False)
)

listing_county_summary.head(20)

,CountyOrParish,new_listings,median_list_price,average_list_price,median_list_price_per_sqft,average_listing_days_on_market
20,Los Angeles,137288,950000.0,1.703722e+06,630.983753,20.284686
37,Riverside,75727,620000.0,7.964622e+05,333.333333,22.501670
41,San Diego,67566,910000.0,1.311613e+06,606.060606,16.321819
31,Orange,51880,1200000.0,1.790212e+06,689.655172,18.453180
40,San Bernardino,51056,539900.0,6.107124e+05,338.148688,20.633168
47,Santa Clara,25848,1499000.0,1.918262e+06,921.902068,15.973808
0,Alameda,25722,998000.0,1.175074e+06,660.654643,16.742166
7,Contra Costa,24645,809990.0,1.118007e+06,517.123842,17.517752
60,Ventura,15359,899000.0,1.271285e+06,530.847618,21.825119
45,San Mateo,9838,1648000.0,2.421454e+06,1004.838911,17.368571


## 18. Listing Segmented Summary by Property Type and Subtype

This summary supports property-level market activity analysis for future Tableau filters.

In [55]:
listing_property_summary = (
    listings.groupby(["PropertyType", "PropertySubType"], dropna=False)
    .agg(
        new_listings=("ListingContractDate", "count"),
        median_list_price=("ListPrice", "median"),
        average_list_price=("ListPrice", "mean"),
        median_list_price_per_sqft=("list_price_per_sqft", "median"),
        average_listing_days_on_market=("listing_days_on_market", "mean")
    )
    .reset_index()
    .sort_values("new_listings", ascending=False)
)

listing_property_summary.head(20)

,PropertyType,PropertySubType,new_listings,median_list_price,average_list_price,median_list_price_per_sqft,average_listing_days_on_market
15,Residential,SingleFamilyResidence,394000,924900.0,1.489540e+06,540.010926,18.979122
4,Residential,Condominium,97715,640000.0,8.117461e+05,582.772544,21.584230
19,Residential,Townhouse,31419,819000.0,9.641405e+05,570.108696,19.493077
10,Residential,ManufacturedOnLand,7581,329900.0,3.766327e+05,234.981764,21.185991
6,Residential,Duplex,3384,982500.0,1.341381e+06,558.251238,19.143026
16,Residential,StockCooperative,1725,374998.0,4.361808e+05,404.347826,19.697391
21,Residential,NaN,1247,810000.0,1.190685e+06,527.466366,20.224539
2,Residential,Cabin,952,299000.0,3.620350e+05,346.877778,23.878151
20,Residential,Triplex,674,1200000.0,1.541109e+06,504.462553,20.563798
11,Residential,MixedUse,492,850000.0,1.594753e+06,484.293194,24.203252


## 19. Listing Sample Output Table

This sample table verifies that the listing-side engineered columns were created correctly.

In [56]:
listing_sample_cols = [
    "ListingContractDate",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "list_year",
    "list_month",
    "list_yrmo",
    "list_price_per_sqft",
    "listing_days_on_market",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "City",
    "PostalCode",
    "ListOfficeName"
]

listing_sample_cols_existing = [col for col in listing_sample_cols if col in listings.columns]
week6_listing_sample_output = listings[listing_sample_cols_existing].head(20)

week6_listing_sample_output

,ListingContractDate,ListPrice,LivingArea,DaysOnMarket,list_year,list_month,list_yrmo,list_price_per_sqft,listing_days_on_market,PropertyType,PropertySubType,CountyOrParish,MLSAreaMajor,City,PostalCode,ListOfficeName
2,2024-01-01,1340000.0,1301.0,127,2024,1,2024-01,1029.976941,127,Residential,Condominium,Los Angeles,C05 - Westwood - Century City,Los Angeles,90067,Rodeo Realty- Brentwood
3,2024-01-24,2500000.0,2788.0,1,2024,1,2024-01,896.700143,1,Residential,SingleFamilyResidence,Orange,LNSLT - Salt Creek,Laguna Niguel,92677,Your Home Sold Guaranteed Realty
4,2024-01-12,3150000.0,3250.0,1,2024,1,2024-01,969.230769,1,Residential,SingleFamilyResidence,Los Angeles,655 - San Marino,San Marino,91108,COMPASS
5,2024-01-20,3090000.0,7456.0,0,2024,1,2024-01,414.431330,0,Residential,SingleFamilyResidence,Los Angeles,616 - Diamond Bar,Diamond Bar,91765,"Signature One Realty Group, Inc"
6,2024-01-12,12725000.0,2321.0,2,2024,1,2024-01,5482.550625,2,Residential,SingleFamilyResidence,Orange,N9 - Lower Newport Bay - Balboa Island,Newport Beach,92662,Pacific Sotheby's Int'l Realty
7,2024-01-30,665000.0,1487.0,2,2024,1,2024-01,447.209146,2,Residential,ManufacturedOnLand,San Diego,92040 - Lakeside,Lakeside,92040,Bryon Thompson Broker
8,2024-01-26,1125000.0,1750.0,2,2024,1,2024-01,642.857143,2,Residential,SingleFamilyResidence,Riverside,321 - Rancho Mirage,Rancho Mirage,92270,The Desert Life Realty
9,2024-01-08,1849000.0,3152.0,3,2024,1,2024-01,586.611675,3,Residential,SingleFamilyResidence,Los Angeles,634 - La Canada Flintridge,La Canada Flintridge,91011,Coldwell Banker Realty
10,2024-01-25,1389000.0,2282.0,5,2024,1,2024-01,608.676599,5,Residential,SingleFamilyResidence,Orange,77 - Anaheim Hills,Anaheim Hills,92808,T.N.G. Real Estate Consultants
11,2024-01-29,1249888.0,1899.0,0,2024,1,2024-01,658.182201,0,Residential,Townhouse,Orange,JS - San Juan South,San Juan Capistrano,92675,Compass


---

# Part C – Save Week 6 Outputs

This section saves the engineered datasets, sample output tables, validation summaries, and segmented summary tables.

In [58]:
# Sold outputs
sold_engineered_output = OUTPUT_DIR / "sold_week6_engineered_metrics.csv"
sold_sample_output = OUTPUT_DIR / "week6_sold_sample_output.csv"
county_output = OUTPUT_DIR / "week6_sold_county_summary.csv"
property_output = OUTPUT_DIR / "week6_sold_property_summary.csv"
mls_area_output = OUTPUT_DIR / "week6_sold_mls_area_summary.csv"
office_output = OUTPUT_DIR / "week6_sold_office_summary.csv"
sold_null_summary_output = OUTPUT_DIR / "week6_sold_metric_null_summary.csv"
sold_metric_summary_output = OUTPUT_DIR / "week6_sold_metric_summary.csv"

# Listing outputs
listing_engineered_output = OUTPUT_DIR / "listing_week6_engineered_metrics.csv"
listing_sample_output = OUTPUT_DIR / "week6_listing_sample_output.csv"
monthly_new_listings_output = OUTPUT_DIR / "week6_monthly_new_listings.csv"
listing_county_output = OUTPUT_DIR / "week6_listing_county_summary.csv"
listing_property_output = OUTPUT_DIR / "week6_listing_property_summary.csv"

# Save sold files
sold.to_csv(sold_engineered_output, index=False)
week6_sold_sample_output.to_csv(sold_sample_output, index=False)
county_summary.to_csv(county_output, index=False)
property_summary.to_csv(property_output, index=False)
mls_area_summary.to_csv(mls_area_output, index=False)
office_summary.to_csv(office_output, index=False)
sold_metric_nulls.to_csv(sold_null_summary_output, index=False)
sold_metric_summary.to_csv(sold_metric_summary_output)

# Save listing files
listings.to_csv(listing_engineered_output, index=False)
week6_listing_sample_output.to_csv(listing_sample_output, index=False)
monthly_new_listings.to_csv(monthly_new_listings_output, index=False)
listing_county_summary.to_csv(listing_county_output, index=False)
listing_property_summary.to_csv(listing_property_output, index=False)

print("Week 6 outputs saved to:", OUTPUT_DIR)
print("Sold outputs:")
for path in [
    sold_engineered_output,
    sold_sample_output,
    county_output,
    property_output,
    mls_area_output,
    office_output,
    sold_null_summary_output,
    sold_metric_summary_output
]:
    print(path)

print("Listing outputs:")
for path in [
    listing_engineered_output,
    listing_sample_output,
    monthly_new_listings_output,
    listing_county_output,
    listing_property_output
]:
    print(path)

Week 6 outputs saved to: /Users/amyliu/Desktop/IDX/data/generated/week6
Sold outputs:
/Users/amyliu/Desktop/IDX/data/generated/week6/sold_week6_engineered_metrics.csv
/Users/amyliu/Desktop/IDX/data/generated/week6/week6_sold_sample_output.csv
/Users/amyliu/Desktop/IDX/data/generated/week6/week6_sold_county_summary.csv
/Users/amyliu/Desktop/IDX/data/generated/week6/week6_sold_property_summary.csv
/Users/amyliu/Desktop/IDX/data/generated/week6/week6_sold_mls_area_summary.csv
/Users/amyliu/Desktop/IDX/data/generated/week6/week6_sold_office_summary.csv
/Users/amyliu/Desktop/IDX/data/generated/week6/week6_sold_metric_null_summary.csv
/Users/amyliu/Desktop/IDX/data/generated/week6/week6_sold_metric_summary.csv
Listing outputs:
/Users/amyliu/Desktop/IDX/data/generated/week6/listing_week6_engineered_metrics.csv
/Users/amyliu/Desktop/IDX/data/generated/week6/week6_listing_sample_output.csv
/Users/amyliu/Desktop/IDX/data/generated/week6/week6_monthly_new_listings.csv
/Users/amyliu/Desktop/IDX/da

## Week 6 Summary

In this notebook, I created the required Week 6 feature-engineered market metrics from both the sold and listing datasets.

For the **sold dataset**, I engineered price ratio, close-to-original-list ratio, price per square foot, days on market, year/month/YrMo variables, listing-to-contract days, and contract-to-close days. I also generated sample output and segmented summaries by county, property type/subtype, MLS area, and office.

For the **listing dataset**, I created listing-side time variables, list price per square foot, listing days on market, monthly new-listings summaries, and segmented summaries by county and property type/subtype.

These outputs prepare the cleaned MLS data for future Tableau dashboard development and market analysis.